# 💰 省钱 SFT 实战：Unsloth 五件套让 27B 模型跑进免费 Colab

> 本教程整理自 [jackrong《Jackrong-llm-finetuning-guide》](https://github.com/R6410418/Jackrong-llm-finetuning-guide) 仓库最有价值的 SFT 教程
> （`train_code/Qwopus3-5-27b-Colab.ipynb`，配套完整指南见 `reference/Qwopus3-5-27b-Colab_complete_guide_to_llm_finetuning.pdf`），
> 按 cook 风格重排并补充"每一处配置省了什么显存"的原理讲解。全部使用开源模型与开源数据，**免费 Colab 单卡即可复现**。

| 项目 | 内容 |
|---|---|
| 🧠 基座模型 | `unsloth/Qwen3.5-27B`（4bit QLoRA 量化加载，单卡可训） |
| 🛠️ 训练框架 | Unsloth + TRL `SFTTrainer`（LoRA r=64，可训练参数 ~1%） |
| 📚 训练数据 | 三源高保真蒸馏数据混合 ~14K 条（nohurry/Opus-4.6 + Roman/claude-opus + Jackrong/700x，见第 6 节） |
| 📏 产出 | LoRA → merged 16bit → GGUF（q4_k_m / q8_0 / bf16 三档） |

**目录**
1. 为什么省：SFT 显存账单与五件套
2. 实验设计总览
3. 环境安装
4. 实验配置
5. 省钱核心 ① 模型侧：4bit QLoRA + LoRA
6. 省钱核心 ② 数据侧：多源归一化管道
7. 省钱核心 ③ 训练侧：8bit 优化器 + 梯度累积 + 检查点瘦身
8. 训练与观察
9. 评估：训练前后对比
10. 保存、合并与 GGUF 导出
11. 附录

## 1️⃣ 为什么省：SFT 显存账单与五件套

全参微调一个 27B 模型的显存账单是"劝退级"的。逐项拆开看，每一处大头都有对应的省钱手段：

| 显存去向 | 全参微调 27B（bf16） | 五件套之后 |
|---|---|---|
| **权重** | 54 GB | ~13.5 GB（4bit NF4 量化） |
| **优化器状态** | 108 GB（AdamW fp32，2×权重） | 只有 LoRA 参数的优化器状态（8bit 再省 4×） |
| **梯度** | 54 GB | 只有 LoRA 梯度 |
| **激活值** | 长序列时最大头 | Unsloth 梯度检查点 -30% 显存 |

**五件套速查**（本教程逐件展开）：

| # | 配置 | 省什么 | 代码位置 | 代价 |
|---|---|---|---|---|
| 1 | `load_in_4bit = True` | 权重显存 4× | 第 5 节 | 精度略降（NF4 实测损失很小） |
| 2 | LoRA（`r=64`，冻结 99% 参数） | 梯度 + 优化器状态两个大头直接消失 | 第 5 节 | 容量上限低于全参微调 |
| 3 | `use_gradient_checkpointing = "unsloth"` | 激活值 -30%，batch 可翻倍 | 第 5 节 | 训练慢 ~20% |
| 4 | `optim = "adamw_8bit"` | 优化器状态 4× | 第 7 节 | 基本无损 |
| 5 | `gradient_accumulation_steps = 6` | 等效 batch ×6 不加显存 | 第 7 节 | 每 6 步才更新一次权重 |

> 💡 这就是"省钱 SFT"的含义：**不是少训练，而是把 27B 的账单砍到免费 Colab 单卡能付得起。**
> jackrong 用这套配置发布的 Qwopus3.5 系列，HF 下载量超过百万——省钱配置与产出质量并不冲突。

## 2️⃣ 实验设计总览

```
                    ┌──────────────────────────────────────────────────────┐
                    │               Qwopus 式省钱 SFT 管线                    │
                    │                                                      │
 三源蒸馏数据 ──▶│ ① 多源 schema → 统一 conversations（<think> 契约）       │
 (nohurry       │ ② qwen3-thinking 模板 → 长度过滤 → 格式校验               │
  Roman 700x)   │ ③ 27B 4bit 加载 + LoRA(r=64)                            │
                │ ④ SFT：8bit 优化器 + 梯度累积（等效 batch 36）            │
                │ ⑤ merged 16bit → GGUF 三档量化                          │
                └──────────────────────────────────────────────────────┘
```

**关键设计决策：**

| 决策 | 选择 | 原因 |
|---|---|---|
| 基座 | `unsloth/Qwen3.5-27B` | Qwopus 官方配方；Unsloth 预量化版免去自己量化 |
| 量化 | 4bit（NF4） | 27B → ~13.5GB，Colab 单卡可训 |
| LoRA | r=64 / α=64 / dropout=0 | r=64 是官方建议档位；α=r 是稳定起点 |
| 序列长度 | 32,768 | 27B 有长上下文能力，长 CoT 数据不截断 |
| 数据 | 三源混合 ~14K | 两源通用推理 + 一源 Jackrong 自产；多样性优于单源 |
| batch | 6 × 累积 6 = 等效 36 | 小步微 batch 保显存，累积保训练稳定性 |
| 学习率 | 2e-4（LoRA 常规档） | 长训降 2e-5（注释原样保留） |

## 3️⃣ 环境安装

> 请准备两把钥匙存进 **Colab Secrets**：`WANDB_API_KEY`（训练可视化）、`HF_TOKEN`（推送模型）。
> 本 cell 复用原 27B notebook 的安装逻辑（uv 加速 + Unsloth 全家桶）。

In [ ]:
import os
from google.colab import drive, userdata
import wandb

drive.mount('/content/drive')
wandb.login(key=userdata.get('WANDB_API_KEY'))
drive_output_path = "/content/drive/MyDrive/Qwen3.5-27B--checkpoints"
os.makedirs(drive_output_path, exist_ok=True)

In [ ]:
%%capture
import importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

## 4️⃣ 实验配置

所有超参数集中在一个 dataclass（本 cell 无 GPU 也能跑，是 notebook 的"配置单"）。
第 7 节的 `SFTConfig` 会从这里取值。

In [1]:
from dataclasses import dataclass, field
import torch

@dataclass
class CheapSFTConfig:
    # ---- 模型 ----
    model_name: str = "unsloth/Qwen3.5-27B"          # 4bit 预量化版；小显存可换 9B/4B（见 fourbit_models 列表）
    max_seq_length: int = 32768                       # 长上下文；显存不足先砍这里

    # ---- LoRA（第 5 节）----
    lora_r: int = 64                                  # 8/16/32/64/128：越大容量越高、越费显存
    lora_alpha: int = 64
    lora_dropout: float = 0.0                         # =0 走 Unsloth 优化路径
    target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "out_proj"])

    # ---- 数据（第 6 节）----
    max_context_window: int = 8192                    # 超过该 token 数的样本直接过滤
    num_samples: dict = field(default_factory=lambda: {
        "ds1": 3900,   # nohurry/Opus-4.6-Reasoning-3000x-filtered
        "ds2": 700,    # Jackrong/Qwen3.5-reasoning-700x
        "ds3": 9633,   # Roman1111111/claude-opus-4.6-10000x
    })
    seed: int = 3407

    # ---- 训练（第 7 节）----
    per_device_train_batch_size: int = 6              # 等效 batch = 6 × 6 = 36
    gradient_accumulation_steps: int = 6
    learning_rate: float = 2e-4                       # 长训练降到 2e-5（原 notebook 注释）
    num_train_epochs: int = 2
    warmup_ratio: float = 0.05
    optim: str = "adamw_8bit"                         # 省钱五件套 ④
    weight_decay: float = 0.001
    save_steps: int = 200
    save_total_limit: int = 1                         # 省钱五件套 ⑤ 的姊妹配置：只留 1 个 checkpoint

cfg = CheapSFTConfig()

# ---- 显存自动调档（opd 风格）：小显存自动降 batch 与序列长度 ----
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory
    if vram < 16 * 2**30:          # < 16GB（如 T4）：降档
        cfg.max_seq_length = 8192
        cfg.per_device_train_batch_size = 2
        print(f"⚠️ 检测到 {vram/2**30:.0f}GB 显存：已自动降档（seq 8K, batch 2）")

torch.manual_seed(cfg.seed)
cfg

CheapSFTConfig(model_name='unsloth/Qwen3.5-27B', max_seq_length=32768, lora_r=64, lora_alpha=64, lora_dropout=0.0, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'out_proj'], max_context_window=8192, num_samples={'ds1': 3900, 'ds2': 700, 'ds3': 9633}, seed=3407, per_device_train_batch_size=6, gradient_accumulation_steps=6, learning_rate=0.0002, num_train_epochs=2, warmup_ratio=0.05, optim='adamw_8bit', weight_decay=0.001, save_steps=200, save_total_limit=1)

## 5️⃣ 省钱核心 ① 模型侧：4bit QLoRA + LoRA

**原理**：`load_in_4bit` 把权重压到 NF4（4 倍省显存）；LoRA 只训练低秩矩阵
`ΔW = A·B`（r=64 时 ~1% 参数），**梯度与优化器状态这两个大头随之消失**。
`use_gradient_checkpointing="unsloth"` 是 Unsloth 特供：比原生省 30% 显存、支持 2 倍 batch。

| 参数 | 值 | 说明（注释来自原 notebook） |
|---|---|---|
| `load_in_4bit` | True | 4bit 量化省显存 |
| `load_in_8bit` | False | 8bit 更准但 2× 显存 |
| `r` | 64 | 越高容量越大、越慢 |
| `lora_alpha` | 64 | LoRA 缩放因子 |
| `lora_dropout` / `bias` | 0 / "none" | Unsloth 优化路径 |
| `use_gradient_checkpointing` | "unsloth" | 省 30% 显存，batch 可翻倍 |

In [ ]:
from unsloth import FastLanguageModel
import torch

# 官方 4bit 预量化模型列表（原 notebook 原样）：按显存挑选，HF 会直接下载
fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit", # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.model_name,
    max_seq_length = cfg.max_seq_length,   # 长上下文！
    load_in_4bit = True,                   # 五件套 ①：4bit 量化省显存
    load_in_8bit = False,                  # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False,               # [NEW!] We have full finetuning now!
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = cfg.lora_r,                        # 五件套 ②：LoRA rank，越高容量越大、越费显存
    target_modules = cfg.target_modules,
    lora_alpha = cfg.lora_alpha,
    lora_dropout = cfg.lora_dropout,       # Supports any, but = 0 is optimized
    bias = "none",                         # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # 五件套 ③：True or "unsloth" for very long context
    random_state = cfg.seed,
    use_rslora = False,                    # We support rank stabilized LoRA
    loftq_config = None,                   # And LoftQ
)

## 6️⃣ 省钱核心 ② 数据侧：多源归一化管道

**数据为什么也省钱**：① 只采 ~14K 条高质量蒸馏数据而非全量（样本效率）；② 超长序列过滤（每条样本的显存峰值都受控）；
③ 第 7 节的 `train_on_responses_only` 只对 assistant 回复算 loss，prompt 部分不浪费算力。

**三种 schema 一张表**（数据集详情见 `food/post-training.md` 第八节）：

| 数据集 | 字段结构 | 适配器 | 坑 |
|---|---|---|---|
| `nohurry/Opus-4.6-Reasoning-3000x-filtered` | `problem / thinking / solution` | `format_ds1` | `thinking` 是裸 CoT，要补 `<think>` 包裹 |
| `Jackrong/Qwen3.5-reasoning-700x` | `conversation: [{from, value}]` | `format_ds2` | 多轮，需保序 + 末轮必须是 assistant |
| `Roman1111111/claude-opus-4.6-10000x` | `messages`（assistant 可带 `reasoning`） | `format_ds3` | ⚠️ `load_dataset` 报 `Feature 'Json' not found`，用 pandas 直读 parquet 绕过 |

**统一契约**：所有 assistant 回复标准化为 `<think>...</think>\n最终答案`，
再用 qwen3-thinking 模板渲染。下面四个函数 + 两个过滤器就是全部核心（已在本机 CPU 用
真实 DeepSeek 数据 + 各 schema 合成样本验证过，见 cell 输出）。

In [2]:
import multiprocessing as mp
import os
import re
from pathlib import Path

from datasets import Dataset, concatenate_datasets, load_dataset
import pandas as pd

RANDOM_SEED = 12181531

# ============================================================
# ② assistant 内容标准化（原 notebook 函数，逐行保留）
# ============================================================
def _strip(x):
    return (x or "").strip()

THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL)

def normalize_assistant_to_think_solution(text: str) -> str:
    """把任意回答统一成 "<think>...</think>\n答案" 结构（Qwopus 格式契约）。"""
    text = _strip(text)
    if not text:
        return "<think></think>\n"
    m = THINK_BLOCK_RE.search(text)
    if m:
        think_block = m.group(0).strip()
        rest = text[m.end():].lstrip()
        return f"{think_block}\n{rest}".rstrip() if rest else f"{think_block}\n"
    return f"<think></think>\n{text}".rstrip()

def build_assistant_with_reasoning(content: str, reasoning: str = "") -> str:
    """content 与 reasoning（隐藏思维链字段）合并；已有 <think> 则只做规范化。"""
    content = _strip(content)
    reasoning = _strip(reasoning)
    if "<think>" in content and "</think>" in content:
        return normalize_assistant_to_think_solution(content)
    if reasoning:
        if content:
            return f"<think>{reasoning}</think>\n{content}"
        return f"<think>{reasoning}</think>\n"
    return normalize_assistant_to_think_solution(content)

def parse_message_item(m):
    """Roman 数据集的 message 可能是 dict 也可能是 JSON 字符串。"""
    if isinstance(m, dict):
        return m
    if isinstance(m, str):
        s = m.strip()
        if not s:
            return None
        try:
            import json as _json
            obj = _json.loads(s)
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None
    return None

# ============================================================
# ① 各源 schema → conversations（原 notebook 的 format_ds1/2/3）
# ============================================================
def format_ds1(examples):
    """nohurry/Opus-4.6-Reasoning-3000x-filtered：problem/thinking/solution。"""
    problems = examples.get("problem", [])
    thinkings = examples.get("thinking", [])
    solutions = examples.get("solution", [])
    out = []
    for p, t, s in zip(problems, thinkings, solutions):
        p, t, s = _strip(p), _strip(t), _strip(s)
        if not p or not s:
            continue
        assistant = f"<think>{t}</think>\n{s}" if t else f"<think></think>\n{s}"
        out.append([{"role": "user", "content": p},
                    {"role": "assistant", "content": assistant}])
    return {"conversations": out}

def format_ds2(examples):
    """Jackrong/Qwen3.5-reasoning-700x：conversation [{from, value}] 多轮。"""
    convos_list = examples.get("conversation", [])
    out = []
    for conv in convos_list:
        if not conv:
            continue
        cleaned = []
        for m in conv:
            frm = (m.get("from") or "").strip()
            val = m.get("value", "")
            if frm == "human":
                cleaned.append({"role": "user", "content": _strip(val)})
            elif frm == "gpt":
                cleaned.append({"role": "assistant",
                                "content": normalize_assistant_to_think_solution(val)})
        if len(cleaned) < 2 or cleaned[-1]["role"] != "assistant":
            continue
        out.append(cleaned)
    return {"conversations": out}

def format_ds3(examples):
    """Roman1111111/claude-opus-4.6-10000x：messages + assistant reasoning。"""
    messages_list = examples.get("messages", [])
    out = []
    for msgs in messages_list:
        if not msgs:
            continue
        parsed_msgs = [pm for pm in (parse_message_item(m) for m in msgs) if pm is not None]
        convo = [m for m in parsed_msgs if m.get("role") != "system"]
        if len(convo) < 2 or convo[-1].get("role") != "assistant":
            continue
        cleaned = []
        for m in convo:
            role = m.get("role")
            content = m.get("content", "")
            reasoning = m.get("reasoning", "")
            if role == "assistant":
                content = build_assistant_with_reasoning(content, reasoning)
            else:
                content = _strip(content)
            if role in ("user", "assistant") and content is not None:
                cleaned.append({"role": role, "content": content})
        if len(cleaned) < 2 or cleaned[-1]["role"] != "assistant":
            continue
        out.append(cleaned)
    return {"conversations": out}

# ============================================================
# ④ 长度过滤 + 格式校验（原 notebook 同名逻辑）
# ============================================================
num_proc = max(1, mp.cpu_count() // 2)

def filter_long_sequences_batched(examples, tok):
    tokenized = tok(examples["text"], truncation=False, padding=False,
                    add_special_tokens=False)["input_ids"]
    return [len(toks) <= cfg.max_context_window for toks in tokenized]

def check_assistant_format(examples):
    convos = examples["conversations"]
    ok = []
    for convo in convos:
        good = True
        for m in convo:
            if m["role"] == "assistant":
                c = m.get("content", "")
                if "<think>" not in c or "</think>" not in c:
                    good = False; break
                if not re.search(r"</think>\n", c):
                    good = False; break
        ok.append(good)
    return {"_ok": ok}

print("✅ 归一化管道函数就绪：format_ds1/2/3 + 长度过滤 + 格式校验")

✅ 归一化管道函数就绪：format_ds1/2/3 + 长度过滤 + 格式校验


In [3]:
# ============================================================
# 本机 CPU 演示（非 Colab 环境走此分支）：用本地缓存的小样本验证整条管道。
# Colab 上会跳过此分支，直接跑下面的"三源配方"。
# ============================================================
IN_COLAB = "COLAB_GPU" in os.environ

if not IN_COLAB:
    from transformers import AutoTokenizer
    import json as _json

    demo_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

    # 真实数据：Jackrong/DeepSeek-V3.2-Exp-reasoning-example（208 行，本地 HF 缓存）
    ds_file = next(Path.home().glob(
        ".cache/huggingface/hub/datasets--Jackrong--DeepSeek-V3.2-Exp-reasoning-example/snapshots/*/deepseek-v3.2-Exp-math.jsonl"))
    rows = [_json.loads(l) for l in ds_file.read_text().splitlines() if l.strip()]
    demo = Dataset.from_dict({
        "question": [r["question"] for r in rows],
        "reasoning": [r["reasoning"] for r in rows],
        "answer": [r["answer-ds-v3.2-Exp"] for r in rows]})

    def format_deepseek(examples):
        out = []
        for q, r, a in zip(examples["question"], examples["reasoning"], examples["answer"]):
            out.append([{"role": "user", "content": _strip(q)},
                        {"role": "assistant", "content": build_assistant_with_reasoning(a, r)}])
        return {"conversations": out}
    demo = demo.map(format_deepseek, batched=True, remove_columns=demo.column_names)

    # 合成样本：其余三种 schema 各 1 条 + 1 条格式反例
    demo = concatenate_datasets([
        demo,
        Dataset.from_dict(format_ds1({
            "problem": ["252 students and 8 teachers go on a field trip……"],
            "thinking": ["Let me work through this step by step. First, total people = 260……"],
            "solution": ["# Solution\n\n## Step 1: 252 + 8 = 260 people……"]})),
        Dataset.from_dict(format_ds3({
            "messages": [[{"role": "system", "content": "You are a helpful AI assistant."},
                          {"role": "user", "content": "Ken created a care package……"},
                          {"role": "assistant", "reasoning": "First find total weight……",
                           "content": "The box weighs 2 pounds."}]]})),
        Dataset.from_dict(format_ds2({
            "conversation": [[{"from": "human", "value": "解释什么是边际效应递减。"},
                              {"from": "gpt", "value": "<think>边际效用随消费量增加而递减……</think>\n边际效应递减是指……"}]]})),
        Dataset.from_dict({"conversations": [[
            {"role": "user", "content": "bad"},
            {"role": "assistant", "content": "no think block here"}]]}),   # 反例：应被格式校验拦截
    ]).shuffle(seed=RANDOM_SEED)

    def formatting_prompts_func(examples):
        return {"text": [demo_tok.apply_chat_template(c, tokenize=False,
                                                      add_generation_prompt=False,
                                                      enable_thinking=False)
                         for c in examples["conversations"]]}
    demo = demo.map(formatting_prompts_func, batched=True)
    _text_tok = getattr(demo_tok, "tokenizer", demo_tok)
    demo = demo.filter(lambda b: filter_long_sequences_batched(b, _text_tok), batched=True)
    _check = demo.map(check_assistant_format, batched=True, remove_columns=demo.column_names)
    _bad = len(_check) - sum(_check["_ok"])
    demo = demo.filter(lambda x: all(
        (m["role"] != "assistant") or ("<think>" in m["content"] and "</think>\n" in m["content"])
        for m in x["conversations"]))

    print(f"本机演示：{len(demo)} 条通过管道（真实 DeepSeek {len(rows)} + 合成 4，"
          f"格式反例 {_bad} 条被正确拦截）")
    print("\n--- 管道产物示例（前 300 字符）---")
    print(demo[0]["text"][:300])

Map:   0%|          | 0/208 [00:00<?, ? examples/s]

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

Filter:   0%|          | 0/212 [00:00<?, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

Filter:   0%|          | 0/139 [00:00<?, ? examples/s]

本机演示：138 条通过管道（真实 DeepSeek 208 + 合成 4，格式反例 1 条被正确拦截）

--- 管道产物示例（前 300 字符）---
<|im_start|>user
Solve the following math problem. Make sure to put the answer (and only answer) inside \boxed{}.

What is the number of ordered pairs $(m,n)$ of positive integers such that $3m^2+8mn+3n^2=383$?<|im_end|>
<|im_start|>assistant
<think>
We are given: "What is the number of ordered pair


In [4]:
# ============================================================
# Colab 三源配方（jackrong 27B notebook 原样）：nohurry + Jackrong 700x + Roman
# ============================================================
if IN_COLAB:
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")

    def load_ds3_via_pandas_parquet():
        # ⚠️ Roman 数据集的已知坑：HF datasets 报 "Feature type 'Json' not found"
        # （旧版序列化格式不兼容）→ 用 pandas 直读转换后的 parquet 绕过
        parquet_path = (
            "hf://datasets/Roman1111111/claude-opus-4.6-10000x"
            "@refs/convert/parquet/default/train/0000.parquet")
        df = pd.read_parquet(parquet_path)
        return Dataset.from_pandas(df, preserve_index=False)

    def load_and_sample(dataset_name, sample_count=None, split="train", subset=None):
        try:
            if subset:
                ds = load_dataset(dataset_name, subset, split=split)
            else:
                ds = load_dataset(dataset_name, split=split)
        except ValueError as e:
            if dataset_name == "Roman1111111/claude-opus-4.6-10000x" and "Feature type 'Json' not found" in str(e):
                ds = load_ds3_via_pandas_parquet()
            else:
                raise
        if sample_count is not None:
            sample_count = min(sample_count, len(ds))
            ds = ds.shuffle(seed=RANDOM_SEED).select(range(sample_count))
        return ds

    ds1 = load_and_sample("nohurry/Opus-4.6-Reasoning-3000x-filtered", cfg.num_samples["ds1"])
    ds2 = load_and_sample("Jackrong/Qwen3.5-reasoning-700x", cfg.num_samples["ds2"])
    ds3 = load_and_sample("Roman1111111/claude-opus-4.6-10000x", cfg.num_samples["ds3"])

    ds1 = ds1.map(format_ds1, batched=True, remove_columns=ds1.column_names)
    ds2 = ds2.map(format_ds2, batched=True, remove_columns=ds2.column_names)
    ds3 = ds3.map(format_ds3, batched=True, remove_columns=ds3.column_names)

    combined_dataset = concatenate_datasets([ds1, ds2, ds3]).shuffle(seed=RANDOM_SEED)

    def formatting_prompts_func(examples):
        convos = examples["conversations"]
        texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
                 for convo in convos]
        return {"text": texts}

    dataset = combined_dataset.map(formatting_prompts_func, batched=True)
    _text_tok = getattr(tokenizer, "tokenizer", tokenizer)
    dataset = dataset.filter(lambda b: filter_long_sequences_batched(b, _text_tok),
                             batched=True, num_proc=num_proc)
    dataset = dataset.filter(lambda x: all(
        (m["role"] != "assistant") or ("<think>" in m["content"] and "</think>\n" in m["content"])
        for m in x["conversations"]))

    print(dataset[0]["text"][:800])
    print(f"数据集就绪：{len(dataset)} 条")

## 7️⃣ 省钱核心 ③ 训练侧：8bit 优化器 + 梯度累积 + 检查点瘦身

- **`optim="adamw_8bit"`（五件套 ④）**：AdamW 优化器状态从 fp32 压到 8bit，4 倍省显存。
- **梯度累积（五件套 ⑤）**：等效 batch = `per_device × grad_accum = 6 × 6 = 36`——
  显存只付 6 的账单，训练稳定性拿 36 的效果。
- **`save_total_limit=1`**：checkpoint 是大文件，只留最新 1 个，直存 Google Drive（防 Colab 实例回收）。
- **`train_on_responses_only`**：把 prompt 部分的 label 置 -100，loss 只在 assistant 回复上计算——
  省下来的算力全部花在刀刃上。

**学习率**：LoRA 微调常用 2e-4（原 notebook 注释：长训练降到 2e-5）。

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,          # Colab 分支产物
    eval_dataset = None,              # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,  # 五件套 ⑤：GA 模拟大 batch
        warmup_ratio = cfg.warmup_ratio,
        num_train_epochs = cfg.num_train_epochs,
        learning_rate = cfg.learning_rate,   # 长训降到 2e-5（原注释）
        logging_steps = 1,
        optim = cfg.optim,                   # 五件套 ④：8bit 优化器
        weight_decay = cfg.weight_decay,
        lr_scheduler_type = "linear",
        seed = cfg.seed,
        save_steps = cfg.save_steps,
        save_total_limit = cfg.save_total_limit,   # 只留 1 个 checkpoint（省磁盘）
        save_strategy = "steps",
        report_to = "wandb",
        output_dir = drive_output_path,
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>",
)

# 检查数据与 label 掩码是否符合预期
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

## 8️⃣ 训练与观察

训练时看三件事：
- **loss 稳定下降**：2e-4 下 LoRA 通常几个 step 内明显下降；
- **wandb 面板**：loss / lr / 每步耗时；
- **显存余量**：如果 OOM，按第 11 节附录 B 降档。

In [ ]:
trainer.train()

In [ ]:
import matplotlib.pyplot as plt

if "trainer" in dir() and hasattr(trainer, "state") and trainer.state.log_history:
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    plt.plot(losses); plt.title("SFT loss（应稳定下降）")
    plt.xlabel("logging step"); plt.grid(alpha=0.3); plt.show()
else:
    print("⚠️ 本机未训练（GPU cell 未执行）；Colab 训练后重跑本 cell 即出曲线")

## 9️⃣ 评估：训练前后对比

在**训练前先跑一次**记录基线，训练后 merge LoRA 再跑一次对比（opd 教程同款做法）。
用 vLLM 批量评估 + greedy 解码，保证前后可比。

In [ ]:
from vllm import LLM, SamplingParams

def vllm_generate_answer(llm, user_text, max_new=400):
    out = llm.generate([user_text], SamplingParams(temperature=0.0, max_tokens=max_new))
    return out[0].outputs[0].text

QUAL_PROMPTS = [
    "用三句话向一个十岁的孩子解释什么是黑洞。",
    "我每天通勤 45 分钟，想利用这段时间学习 Python 编程。请给我制定一个为期四周的学习计划。",
    "如果所有的猫都会飞，而咪咪是一只猫，那么咪咪会飞吗？请解释你的推理过程。",
]

# ---- 训练前基线（训练前先单独跑这几行）----
llm = LLM(model="Qwen/Qwen3.5-27B", dtype="bfloat16",
          gpu_memory_utilization=0.9, max_model_len=2048)
baseline_answers = [vllm_generate_answer(llm, q) for q in QUAL_PROMPTS]
for q, a in zip(QUAL_PROMPTS, baseline_answers):
    print(f"【问题】{q}\n【训练前回答】{a[:300]}\n{'-'*60}")

In [ ]:
# ---- 训练后评估（训练 + merge 后跑这几行）----
import gc
del llm; gc.collect(); import torch as _t; _t.cuda.empty_cache()

merged = model.merge_and_unload()
merged.save_pretrained("./qwopus_merged")
tokenizer.save_pretrained("./qwopus_merged")
del merged; gc.collect(); _t.cuda.empty_cache()

llm = LLM(model="./qwopus_merged", dtype="bfloat16",
          gpu_memory_utilization=0.9, max_model_len=2048)
for q, before in zip(QUAL_PROMPTS, baseline_answers):
    after = vllm_generate_answer(llm, q)
    print(f"【问题】{q}")
    print(f"\n🔹 训练前：{before[:300]}")
    print(f"\n🔸 训练后：{after[:300]}")
    print("=" * 70)

## 🔟 保存、合并与 GGUF 导出（Qwopus 风格收尾）

1. **保存 LoRA**：适配器只有几十 MB，最轻量；
2. **合并 16bit**：`push_to_hub_merged(save_method="merged_16bit")` 一次完成合并+上传；
3. **GGUF 三档量化**：`q4_k_m / q8_0 / bf16`，覆盖 llama.cpp / Ollama / LM Studio 本地部署。

In [ ]:
# ---- 1. 保存 LoRA ----
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

In [ ]:
# ---- 2. 合并 16bit 并推送 ----
from huggingface_hub import whoami
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
username = whoami(token=hf_token)["name"]
repo_id = f"{username}/Qwopus3.5-27B"

model.push_to_hub_merged(repo_id, tokenizer, save_method="merged_16bit", token=hf_token)
print(f"Uploaded to https://huggingface.co/{repo_id}")

In [ ]:
# ---- 3. GGUF 三档量化并推送 ----
model.push_to_hub_gguf(
    f"{username}/Qwopus3.5-27B-GGUF",
    tokenizer,
    quantization_method=["q4_k_m", "q8_0", "bf16"],
    token=hf_token,
)

## 1️⃣1️⃣ 附录

### A. 不同硬件的配置建议

| 硬件 | 模型 | seq | batch×累积 | LoRA | 预期 |
|---|---|---|---|---|---|
| Colab T4（16GB） | Qwen3-4B | 8K | 2×4 | r=32 | 快速跑通全流程 |
| Colab A100（40GB） | Qwen3.5-27B | 32K | 6×6 | r=64 | 本教程默认配置 |
| Kaggle 双卡 | Qwen3.5-35B-A3B | 8K | 2×4 | r=32 | MoE 版（另见 35B notebook） |

### B. 故障排查

| 症状 | 可能原因 | 处理 |
|---|---|---|
| OOM | seq 32K × batch 6 超显存 | 先砍 `max_seq_length`，再砍 batch（config cell 会自动降档） |
| `Feature type 'Json' not found` | Roman 数据集旧格式 | 已内置 parquet 绕过（第 6 节） |
| loss 不降 | lr 过高 / 数据格式错乱 | lr 降 2e-5；检查第 6 节 demo 输出 |
| Colab 断连丢 checkpoint | 实例回收 | `save_total_limit=1` 已直存 Drive，重新挂载即可 |

### C. 参考

1. jackrong《Jackrong-llm-finetuning-guide》—— 本教程代码来源（[train_code/Qwopus3-5-27b-Colab.ipynb](../../Jackrong-llm-finetuning-guide/train_code/Qwopus3-5-27b-Colab.ipynb)）
2. `reference/Qwopus3-5-27b-Colab_complete_guide_to_llm_finetuning.pdf` —— 官方完整指南
3. `food/post-training.md` —— 24 个 Jackrong 数据集 + 5 个第三方数据集清单（含本教程三源详情）
4. 数据归一化管道已在本机 CPU 用真实数据验证（本 notebook 第 6 节 cell 输出）

---

🎉 **上篇结束**。你已掌握 Qwopus 式省钱 SFT 的完整五件套。中篇（R1-Zero GRPO）见 `cook_r1_grpo.ipynb`，下篇（缝合+愈合）见 `cook_merge_heal.ipynb`。